<a href="https://colab.research.google.com/github/JustinRSK/2025_ML_EES/blob/main/ML_Final_project/2_ML_Final_Project_GC_Justin_Knight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Grande Cariçaie vegetation classification and 2011–2025 comparison

## Machine Learning for Earth & Environmental Sciences

### Objectives

The main objective of this analysis is to reconstruct the vegetation of two Grande
Cariçaie reserves for which no 2010–2011 vegetation labels are available.

A Random Forest classifier is trained using vegetation labels from seven other reserves
and evaluated using leave-one-reserve-out spatial cross-validation.

The analysis has four objectives:

1. Test whether vegetation can be predicted in a completely unseen reserve.
2. Predict vegetation in the two reserves lacking 2010–2011 labels.
3. Produce a model-based estimate of vegetation distribution in 2025.
4. Identify high-confidence candidate vegetation changes between 2011 and 2025.

## Vegetation classes

The original vegetation map contains six broad classes. Preliminary six-class modelling
showed high performance for water, wetland vegetation and forest, but poor discrimination
of rare agricultural and constructed classes.

The final classification therefore uses four ecologically interpretable classes:

1. Eaux libres
2. Rivages et lieux humides
3. Végétation ouverte
   - Pelouses et prairies
   - Plantations, champs et cultures
4. Forêts

"Milieux construits" is excluded because it is rare within the reserves and showed very
poor spatial generalisation at Landsat's 30 m resolution.

## Validation

Random pixels are not randomly divided into training and test sets because neighbouring
pixels are spatially autocorrelated.

Instead, one complete reserve is excluded at a time. The model is trained on the remaining
labelled reserves and tested on the unseen reserve.

This mimics the actual application of the model to the two reserves lacking vegetation
labels.

## 2025 interpretation

The 2025 map is a model-based estimate rather than an independently validated vegetation
survey.

Differences between 2011 and 2025 are therefore interpreted as candidate vegetation
changes and not automatically as confirmed ecological transitions.

In [1]:
# ============================================================
# 1. IMPORTS AND PROJECT PATHS
# ============================================================

%pip install -q rasterio

from pathlib import Path
import subprocess
import sys
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    cohen_kappa_score,
    f1_score,
    make_scorer
)
from sklearn.model_selection import (
    GroupKFold,
    GridSearchCV,
    LeaveOneGroupOut
)


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# ------------------------------------------------------------
# GitHub repository
# ------------------------------------------------------------

REPO_URL = "https://github.com/JustinRSK/2025_ML_EES.git"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:

    REPO_DIR = Path("/content/2025_ML_EES")

    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", REPO_URL, str(REPO_DIR)],
            check=True
        )

    PROJECT_DIR = REPO_DIR / "ML_Final_project"

else:

    cwd = Path.cwd()

    if cwd.name == "ML_Final_project":
        PROJECT_DIR = cwd

    elif (cwd / "ML_Final_project").exists():
        PROJECT_DIR = cwd / "ML_Final_project"

    else:
        PROJECT_DIR = cwd


# ------------------------------------------------------------
# Data directory
# ------------------------------------------------------------

possible_data_dirs = [
    PROJECT_DIR / "data",
    PROJECT_DIR
]

required_files = [
    "GC_predictors_2011.tif",
    "GC_predictors_2025.tif",
    "vegetation_class_2011.tif",
    "reserve_id.tif"
]

DATA_DIR = None

for candidate in possible_data_dirs:

    if all((candidate / f).exists() for f in required_files):
        DATA_DIR = candidate
        break


if DATA_DIR is None:

    raise FileNotFoundError(
        "The four TIFF files were not found. "
        "Place them in ML_Final_project/data/."
    )


OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURE_DIR = PROJECT_DIR / "figures"

OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)


print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

Data directory: /content/2025_ML_EES/ML_Final_project
Output directory: /content/2025_ML_EES/ML_Final_project/outputs


In [2]:
# ============================================================
# 2. CLASS AND PREDICTOR DEFINITIONS
# ============================================================


# Original raster bands
ALL_FEATURES = [
    "blue",
    "green",
    "red",
    "nir",
    "swir1",
    "swir2",
    "NDVI",
    "NDMI",
    "MNDWI",
    "NBR",
    "elevation",
    "slope"
]


# ------------------------------------------------------------
# Features used in the final model
# ------------------------------------------------------------
#
# We prioritise indices that represent vegetation, moisture
# and water conditions and are relatively transferable between
# Landsat generations.
#
# Raw reflectance bands are excluded from the final temporal
# model to reduce direct Landsat-5 vs Landsat-8/9 differences.
# ------------------------------------------------------------

MODEL_FEATURES = [
    "blue",
    "green",
    "red",
    "nir",
    "swir1",
    "swir2",
    "NDVI",
    "NDMI",
    "MNDWI",
    "NBR",
    "elevation",
    "slope"
]

# Final four-class system
CLASS_NAMES = {
    1: "Eaux libres",
    2: "Rivages et lieux humides",
    3: "Végétation ouverte",
    4: "Forêts"
}


RESERVE_NAMES = {
    1: "Grèves de Cheseaux",
    2: "Baie d'Yvonand",
    3: "Cheyres",
    4: "Grèves de la Corbière",
    5: "Grèves d'Ostende",
    6: "Grèves de la Motte",
    7: "Cudrefin",
    8: "Fanel neuchâtelois",
    9: "Vernes et Tuileries"
}


PIXEL_AREA_HA = 30 * 30 / 10000


print("Final model predictors:")
print(MODEL_FEATURES)

Final model predictors:
['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'NDVI', 'NDMI', 'MNDWI', 'NBR', 'elevation', 'slope']


In [3]:
# ============================================================
# 3. LOAD RASTER DATA
# ============================================================


def read_predictors(path):

    with rasterio.open(path) as src:

        data = (
            src.read(masked=True)
            .filled(np.nan)
            .astype(np.float32)
        )

        metadata = {
            "profile": src.profile.copy(),
            "transform": src.transform,
            "crs": src.crs,
            "width": src.width,
            "height": src.height
        }

    return data, metadata


def read_label(path):

    with rasterio.open(path) as src:

        data = (
            src.read(1, masked=True)
            .filled(0)
            .astype(np.int16)
        )

        metadata = {
            "transform": src.transform,
            "crs": src.crs,
            "width": src.width,
            "height": src.height
        }

    return data, metadata


predictors_2011, meta_2011 = read_predictors(
    DATA_DIR / "GC_predictors_2011.tif"
)

predictors_2025, meta_2025 = read_predictors(
    DATA_DIR / "GC_predictors_2025.tif"
)

vegetation_original, meta_class = read_label(
    DATA_DIR / "vegetation_class_2011.tif"
)

reserve, meta_reserve = read_label(
    DATA_DIR / "reserve_id.tif"
)


print("2011:", predictors_2011.shape)
print("2025:", predictors_2025.shape)
print("Vegetation:", vegetation_original.shape)
print("Reserve:", reserve.shape)

2011: (12, 832, 1084)
2025: (12, 832, 1084)
Vegetation: (832, 1084)
Reserve: (832, 1084)


In [4]:
# ============================================================
# 4. GRID QUALITY CONTROL
# ============================================================

assert predictors_2011.shape == predictors_2025.shape

assert predictors_2011.shape[1:] == vegetation_original.shape

assert predictors_2011.shape[1:] == reserve.shape

assert meta_2011["crs"] == meta_2025["crs"]
assert meta_2011["crs"] == meta_class["crs"]
assert meta_2011["crs"] == meta_reserve["crs"]

assert meta_2011["transform"] == meta_2025["transform"]
assert meta_2011["transform"] == meta_class["transform"]
assert meta_2011["transform"] == meta_reserve["transform"]


print("✓ All rasters have the same grid.")
print("CRS:", meta_2011["crs"])
print(
    "Dimensions:",
    meta_2011["width"],
    "x",
    meta_2011["height"]
)

✓ All rasters have the same grid.
CRS: EPSG:2056
Dimensions: 1084 x 832


In [5]:
# ============================================================
# 5. CREATE FINAL FOUR-CLASS VEGETATION LABEL
# ============================================================

vegetation = np.zeros_like(
    vegetation_original,
    dtype=np.uint8
)


# 1 = Eaux libres
vegetation[
    vegetation_original == 1
] = 1


# 2 = Rivages et lieux humides
vegetation[
    vegetation_original == 2
] = 2


# 3 = Végétation ouverte
#
# Original:
# 3 = Pelouses et prairies
# 5 = Plantations / champs / cultures

vegetation[
    (vegetation_original == 3)
    |
    (vegetation_original == 5)
] = 3


# 4 = Forêts
vegetation[
    vegetation_original == 4
] = 4


# Original class 6 = Milieux construits
# remains 0 and is excluded.


print("Classes retained:")
print(np.unique(vegetation))

Classes retained:
[0 1 2 3 4]


In [6]:
# ============================================================
# 6. SELECT MODEL FEATURES
# ============================================================

feature_indices = [
    ALL_FEATURES.index(feature)
    for feature in MODEL_FEATURES
]


p2011 = predictors_2011[
    feature_indices
]

p2025 = predictors_2025[
    feature_indices
]


print("Model stack shape:", p2011.shape)

Model stack shape: (12, 832, 1084)


In [7]:
# ============================================================
# 7. BUILD TRAINING DATA
# ============================================================

valid_predictors = np.isfinite(
    p2011
).all(axis=0)


training_mask = (
    valid_predictors
    &
    (vegetation > 0)
    &
    (reserve > 0)
)


X = p2011[:, training_mask].T

y = vegetation[
    training_mask
].astype(int)

groups = reserve[
    training_mask
].astype(int)


labelled_reserves = sorted(
    np.unique(groups)
)

all_reserves = sorted(
    np.unique(reserve[reserve > 0])
)

unlabelled_reserves = [
    r for r in all_reserves
    if r not in labelled_reserves
]


print("Training pixels:", len(y))
print("Labelled reserves:", labelled_reserves)
print("Unlabelled target reserves:", unlabelled_reserves)

Training pixels: 24084
Labelled reserves: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Unlabelled target reserves: [np.int16(8), np.int16(9)]


In [8]:
# ============================================================
# 8. TRAINING DATA SUMMARY
# ============================================================

training_df = pd.DataFrame({
    "class_id": y,
    "reserve_id": groups
})


class_table = pd.crosstab(
    training_df["reserve_id"],
    training_df["class_id"]
)


class_table.index = [
    RESERVE_NAMES[i]
    for i in class_table.index
]

class_table.columns = [
    CLASS_NAMES[i]
    for i in class_table.columns
]


display(class_table)


class_table.to_csv(
    OUTPUT_DIR / "training_pixels_by_reserve_and_class.csv"
)

,Eaux libres,Rivages et lieux humides,Végétation ouverte,Forêts
Grèves de Cheseaux,1097,812,76,1074
Baie d'Yvonand,2249,447,25,712
Cheyres,773,802,86,944
Grèves de la Corbière,972,679,12,1202
Grèves d'Ostende,1987,2236,138,1020
Grèves de la Motte,966,1567,24,1498
Cudrefin,864,678,203,941


In [9]:
# ============================================================
# 9. RANDOM FOREST CONFIGURATION
# ============================================================

# Number of trees is fixed.
# Increasing this mainly improves stability rather than complexity.
base_model = RandomForestClassifier(
    n_estimators=600,
    random_state=RANDOM_STATE,
    n_jobs=-1
)


# ------------------------------------------------------------
# Hyperparameter search
# ------------------------------------------------------------
#
# Végétation ouverte is strongly under-represented.
#
# We therefore test:
# - standard balanced RF weighting
# - stronger explicit weights for class 3
#
# The final choice is still made using grouped CV.
# ------------------------------------------------------------

PARAM_GRID = {

    "max_features": [
        "sqrt",
        0.6
    ],

    "min_samples_leaf": [
        1,
        2,
        4
    ],

    "max_depth": [
        20,
        None
    ],

    "class_weight": [
        "balanced_subsample",

        {1: 1, 2: 1, 3: 20, 4: 1},

        {1: 1, 2: 1, 3: 30, 4: 1}
    ]
}


# ------------------------------------------------------------
# Standard macro F1
# ------------------------------------------------------------

macro_f1_scorer = make_scorer(
    f1_score,
    average="macro",
    zero_division=0
)


# ------------------------------------------------------------
# F1 specifically for Végétation ouverte (class 3)
# ------------------------------------------------------------

def open_vegetation_f1(
    y_true,
    y_pred
):

    return f1_score(
        y_true == 3,
        y_pred == 3,
        zero_division=0
    )


open_f1_scorer = make_scorer(
    open_vegetation_f1
)


# ------------------------------------------------------------
# Model-selection score
# ------------------------------------------------------------
#
# We still prioritize overall performance,
# but give additional importance to the weak class.
#
# 75% = overall macro F1
# 25% = F1 of Végétation ouverte
# ------------------------------------------------------------

def priority_metric(
    y_true,
    y_pred
):

    macro = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    open_f1 = open_vegetation_f1(
        y_true,
        y_pred
    )

    return (
        0.75 * macro
        +
        0.25 * open_f1
    )


priority_scorer = make_scorer(
    priority_metric
)


SCORING = {
    "priority": priority_scorer,
    "macro_f1": macro_f1_scorer,
    "open_f1": open_f1_scorer,
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy"
}

In [ ]:
# ============================================================
# 10. SPATIAL CROSS-VALIDATION
# ============================================================

outer_cv = LeaveOneGroupOut()

classes = np.array(
    sorted(CLASS_NAMES)
)

# Out-of-fold predictions:
# every labelled pixel will be predicted once when its
# entire reserve is held out.
oof_prediction = np.zeros(
    len(y),
    dtype=np.uint8
)

oof_confidence = np.full(
    len(y),
    np.nan,
    dtype=np.float32
)

fold_results = []
pfi_results = []


# ============================================================
# OUTER CROSS-VALIDATION LOOP
# ============================================================

for fold, (train_idx, test_idx) in enumerate(
    outer_cv.split(X, y, groups),
    start=1
):

    # --------------------------------------------------------
    # Identify held-out reserve
    # --------------------------------------------------------

    held_out_id = int(
        np.unique(groups[test_idx])[0]
    )

    held_out_name = RESERVE_NAMES[
        held_out_id
    ]


    # --------------------------------------------------------
    # Split data
    # --------------------------------------------------------

    X_train = X[train_idx]
    y_train = y[train_idx]
    groups_train = groups[train_idx]

    X_test = X[test_idx]
    y_test = y[test_idx]


    # ========================================================
    # INNER CROSS-VALIDATION
    # ========================================================
    #
    # Hyperparameters are chosen using ONLY the training
    # reserves. The held-out reserve remains completely unseen.
    # ========================================================

    inner_cv = StratifiedGroupKFold(
        n_splits=3,
        shuffle=True,
        random_state=RANDOM_STATE + fold
    )


    search = GridSearchCV(
        estimator=base_model,
        param_grid=PARAM_GRID,
        scoring=SCORING,
        refit="priority",
        cv=inner_cv,
        n_jobs=-1,
        return_train_score=True
    )


    search.fit(
        X_train,
        y_train,
        groups=groups_train
    )


    model = search.best_estimator_


    # ========================================================
    # PREDICT HELD-OUT RESERVE
    # ========================================================

    y_pred = model.predict(
        X_test
    )

    probabilities = model.predict_proba(
        X_test
    )

    confidence = probabilities.max(
        axis=1
    )


    # Store out-of-fold predictions

    oof_prediction[
        test_idx
    ] = y_pred

    oof_confidence[
        test_idx
    ] = confidence


    # ========================================================
    # OUTER TEST METRICS
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )


    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore"
        )

        balanced_accuracy = balanced_accuracy_score(
            y_test,
            y_pred
        )


    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )


    open_f1 = open_vegetation_f1(
        y_test,
        y_pred
    )


    kappa = cohen_kappa_score(
        y_test,
        y_pred
    )


    # ========================================================
    # INNER TRAINING / VALIDATION METRICS
    # ========================================================

    best_idx = search.best_index_


    inner_train_macro_f1 = (
        search.cv_results_[
            "mean_train_macro_f1"
        ][best_idx]
    )


    inner_validation_macro_f1 = (
        search.cv_results_[
            "mean_test_macro_f1"
        ][best_idx]
    )


    inner_validation_open_f1 = (
        search.cv_results_[
            "mean_test_open_f1"
        ][best_idx]
    )


    # ========================================================
    # SAVE RESULTS FOR THIS FOLD
    # ========================================================

    fold_results.append({

        "fold": fold,

        "held_out_reserve":
            held_out_name,

        "accuracy":
            accuracy,

        "balanced_accuracy":
            balanced_accuracy,

        "macro_f1":
            macro_f1,

        "open_vegetation_f1":
            open_f1,

        "cohen_kappa":
            kappa,

        "inner_train_macro_f1":
            inner_train_macro_f1,

        "inner_validation_macro_f1":
            inner_validation_macro_f1,

        "inner_validation_open_f1":
            inner_validation_open_f1,

        "best_parameters":
            str(search.best_params_)
    })


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print("=" * 70)

    print(
        f"Fold {fold}: {held_out_name}"
    )

    print(
        f"Accuracy:                    {accuracy:.3f}"
    )

    print(
        f"Balanced accuracy:           {balanced_accuracy:.3f}"
    )

    print(
        f"Macro F1:                    {macro_f1:.3f}"
    )

    print(
        f"Open vegetation F1:          {open_f1:.3f}"
    )

    print(
        f"Cohen's kappa:               {kappa:.3f}"
    )

    print(
        f"Inner validation macro F1:   "
        f"{inner_validation_macro_f1:.3f}"
    )

    print(
        f"Inner validation open F1:    "
        f"{inner_validation_open_f1:.3f}"
    )

    print(
        "Best parameters:",
        search.best_params_
    )


    # ========================================================
    # PERMUTATION FEATURE IMPORTANCE
    # ========================================================
    #
    # PFI is calculated ONLY on the held-out reserve.
    # This satisfies the requirement that importance is
    # evaluated using independent test data.
    # ========================================================

    if len(X_test) > 5000:

        rng = np.random.default_rng(
            RANDOM_STATE + held_out_id
        )

        idx = rng.choice(
            len(X_test),
            size=5000,
            replace=False
        )

        X_pfi = X_test[idx]
        y_pfi = y_test[idx]

    else:

        X_pfi = X_test
        y_pfi = y_test


    pfi = permutation_importance(
        model,
        X_pfi,
        y_pfi,
        scoring=macro_f1_scorer,
        n_repeats=10,
        random_state=RANDOM_STATE + held_out_id,
        n_jobs=-1
    )


    for feature, importance in zip(
        MODEL_FEATURES,
        pfi.importances_mean
    ):

        pfi_results.append({
            "fold": fold,
            "reserve": held_out_name,
            "feature": feature,
            "importance": importance
        })


print("\nSpatial cross-validation completed.")

Fold 1: Grèves de Cheseaux
Accuracy:                    0.883
Balanced accuracy:           0.685
Macro F1:                    0.684
Open vegetation F1:          0.058
Cohen's kappa:               0.827
Inner validation macro F1:   0.806
Inner validation open F1:    0.521
Best parameters: {'class_weight': 'balanced_subsample', 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 4}


/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [ ]:
# ============================================================
# 11. CROSS-VALIDATION RESULTS
# ============================================================

cv_results = pd.DataFrame(
    fold_results
)


display(
    cv_results
)


print("\nMEAN SPATIAL CROSS-VALIDATION PERFORMANCE")


summary = (
    cv_results[
        [
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "cohen_kappa"
        ]
    ]
    .agg(["mean", "std"])
    .T
)


display(
    summary.round(3)
)


cv_results.to_csv(
    OUTPUT_DIR / "cross_validation_by_reserve.csv",
    index=False
)

summary.to_csv(
    OUTPUT_DIR / "cross_validation_summary.csv"
)

In [ ]:
# ============================================================
# 12. OUT-OF-FOLD CLASSIFICATION REPORT
# ============================================================

class_labels = [
    CLASS_NAMES[c]
    for c in classes
]


report = classification_report(
    y,
    oof_prediction,
    labels=classes,
    target_names=class_labels,
    output_dict=True,
    zero_division=0
)


report_df = pd.DataFrame(
    report
).T


display(
    report_df.round(3)
)


report_df.to_csv(
    OUTPUT_DIR / "classification_report.csv"
)

In [ ]:
# ============================================================
# 13. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y,
    oof_prediction,
    labels=classes,
    normalize="true"
)


fig, ax = plt.subplots(
    figsize=(8, 7)
)


display_cm = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_labels
)


display_cm.plot(
    ax=ax,
    values_format=".2f",
    colorbar=False,
    xticks_rotation=35
)


ax.set_title(
    "Leave-one-reserve-out confusion matrix\n"
    "(normalised by reference class)"
)


plt.tight_layout()


plt.savefig(
    FIGURE_DIR / "confusion_matrix_final.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()

In [ ]:
# ============================================================
# 14. PERMUTATION FEATURE IMPORTANCE
# ============================================================

pfi_df = pd.DataFrame(
    pfi_results
)


pfi_summary = (
    pfi_df
    .groupby("feature")["importance"]
    .agg(
        mean="mean",
        std="std"
    )
    .sort_values(
        "mean",
        ascending=False
    )
)


display(
    pfi_summary.round(3)
)


pfi_df.to_csv(
    OUTPUT_DIR / "PFI_by_reserve.csv",
    index=False
)

pfi_summary.to_csv(
    OUTPUT_DIR / "PFI_summary.csv"
)


plot_pfi = pfi_summary.sort_values(
    "mean"
)


fig, ax = plt.subplots(
    figsize=(7, 5)
)


ax.barh(
    plot_pfi.index,
    plot_pfi["mean"]
)


ax.errorbar(
    plot_pfi["mean"],
    plot_pfi.index,
    xerr=plot_pfi["std"],
    fmt="none",
    capsize=3
)


ax.axvline(
    0,
    linewidth=1
)


ax.set_xlabel(
    "Decrease in macro F1 after permutation"
)

ax.set_title(
    "Permutation feature importance\n"
    "Mean ± SD across held-out reserves"
)


plt.tight_layout()


plt.savefig(
    FIGURE_DIR / "permutation_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()

In [ ]:
# ============================================================
# 15. PREDICTION CONFIDENCE
# ============================================================

correct = (
    oof_prediction == y
)


confidence_rows = []


for threshold in np.arange(
    0.50,
    0.96,
    0.05
):

    selected = (
        oof_confidence >= threshold
    )


    if selected.sum() == 0:
        continue


    empirical_accuracy = correct[
        selected
    ].mean()


    coverage = selected.mean()


    confidence_rows.append({
        "threshold": threshold,
        "empirical_accuracy": empirical_accuracy,
        "coverage": coverage,
        "n_pixels": selected.sum()
    })


confidence_table = pd.DataFrame(
    confidence_rows
)


display(
    confidence_table.round(3)
)


# ------------------------------------------------------------
# Choose the lowest confidence threshold giving approximately
# 90% empirical accuracy while retaining at least 10% of data.
# ------------------------------------------------------------

acceptable = confidence_table[
    (confidence_table["empirical_accuracy"] >= 0.90)
    &
    (confidence_table["coverage"] >= 0.10)
]


if len(acceptable) > 0:

    CONFIDENCE_THRESHOLD = float(
        acceptable.iloc[0]["threshold"]
    )

else:

    # Conservative fallback
    CONFIDENCE_THRESHOLD = 0.80


print(
    "\nConfidence threshold used for candidate change:",
    CONFIDENCE_THRESHOLD
)


confidence_table.to_csv(
    OUTPUT_DIR / "prediction_confidence_validation.csv",
    index=False
)

In [ ]:
# ============================================================
# 16. FINAL MODEL
# ============================================================

final_cv = LeaveOneGroupOut()

final_search = GridSearchCV(
    estimator=base_model,
    param_grid=PARAM_GRID,
    scoring=SCORING,
    refit="priority",
    cv=final_cv,
    n_jobs=-1,
    return_train_score=True
)


final_search.fit(
    X,
    y,
    groups=groups
)

final_model = (
    final_search.best_estimator_
)

print("Final parameters:")
print(
    final_search.best_params_
)


print(
    "\nFinal model trained on",
    len(y),
    "pixels from",
    len(labelled_reserves),
    "labelled reserves."
)


joblib.dump(
    {
        "model": final_model,
        "features": MODEL_FEATURES,
        "classes": CLASS_NAMES,
        "confidence_threshold": CONFIDENCE_THRESHOLD
    },

    OUTPUT_DIR / "Grande_Caricaie_RF_final.joblib"
)

In [ ]:
# ============================================================
# 17. RASTER PREDICTION FUNCTIONS
# ============================================================


def predict_raster(
    model,
    stack,
    chunk_size=100000
):

    n_features, rows, cols = stack.shape


    flat = (
        stack
        .reshape(n_features, -1)
        .T
    )


    valid = np.isfinite(
        flat
    ).all(axis=1)


    valid_idx = np.flatnonzero(
        valid
    )


    predicted = np.zeros(
        flat.shape[0],
        dtype=np.uint8
    )


    confidence = np.full(
        flat.shape[0],
        np.nan,
        dtype=np.float32
    )


    for start in range(
        0,
        len(valid_idx),
        chunk_size
    ):

        idx = valid_idx[
            start:start + chunk_size
        ]


        X_batch = flat[idx]


        probabilities = model.predict_proba(
            X_batch
        )


        best = np.argmax(
            probabilities,
            axis=1
        )


        predicted[idx] = model.classes_[
            best
        ]


        confidence[idx] = probabilities[
            np.arange(len(idx)),
            best
        ]


    return (
        predicted.reshape(rows, cols),
        confidence.reshape(rows, cols)
    )



def save_class_raster(
    array,
    path,
    profile
):

    p = profile.copy()


    p.update(
        count=1,
        dtype="uint8",
        nodata=0,
        compress="lzw"
    )


    with rasterio.open(
        path,
        "w",
        **p
    ) as dst:

        dst.write(
            array.astype(np.uint8),
            1
        )



def save_float_raster(
    array,
    path,
    profile
):

    nodata = -9999.0


    output = np.where(
        np.isfinite(array),
        array,
        nodata
    ).astype(np.float32)


    p = profile.copy()


    p.update(
        count=1,
        dtype="float32",
        nodata=nodata,
        compress="lzw"
    )


    with rasterio.open(
        path,
        "w",
        **p
    ) as dst:

        dst.write(
            output,
            1
        )

In [ ]:
# ============================================================
# 18. PREDICT 2011 AND 2025
# ============================================================

print("Predicting 2011...")


prediction_2011, confidence_2011 = predict_raster(
    final_model,
    p2011
)


print("Predicting 2025...")


prediction_2025, confidence_2025 = predict_raster(
    final_model,
    p2025
)


# Restrict analysis to the nine reserves.

reserve_mask = (
    reserve > 0
)


prediction_2011[
    ~reserve_mask
] = 0

prediction_2025[
    ~reserve_mask
] = 0


confidence_2011[
    ~reserve_mask
] = np.nan

confidence_2025[
    ~reserve_mask
] = np.nan


print("Predictions completed.")

In [ ]:
# ============================================================
# 19. PREDICT THE TWO UNLABELLED 2011 RESERVES
# ============================================================

target_mask = np.isin(
    reserve,
    unlabelled_reserves
)


prediction_targets_2011 = np.where(
    target_mask,
    prediction_2011,
    0
)


confidence_targets_2011 = np.where(
    target_mask,
    confidence_2011,
    np.nan
)


save_class_raster(
    prediction_targets_2011,
    OUTPUT_DIR / "vegetation_2011_reserves_8_9.tif",
    meta_2011["profile"]
)


save_float_raster(
    confidence_targets_2011,
    OUTPUT_DIR / "confidence_2011_reserves_8_9.tif",
    meta_2011["profile"]
)


print(
    "2011 vegetation predictions for reserves 8 and 9 saved."
)

In [ ]:
# ============================================================
# 20. COMPLETE 2011 BASELINE
# ============================================================

baseline_2011 = np.zeros_like(
    prediction_2011
)


baseline_source = np.zeros_like(
    prediction_2011
)


# Reference map where available.
observed = (
    vegetation > 0
)


baseline_2011[
    observed
] = vegetation[
    observed
]


baseline_source[
    observed
] = 1


# RF prediction only for the two unlabeled reserves.
baseline_2011[
    target_mask
] = prediction_2011[
    target_mask
]


baseline_source[
    target_mask
] = 2


save_class_raster(
    baseline_2011,
    OUTPUT_DIR / "vegetation_complete_2011.tif",
    meta_2011["profile"]
)


save_class_raster(
    baseline_source,
    OUTPUT_DIR / "vegetation_2011_source.tif",
    meta_2011["profile"]
)

In [ ]:
# ============================================================
# 21. SAVE 2025 VEGETATION MAP
# ============================================================

save_class_raster(
    prediction_2025,
    OUTPUT_DIR / "vegetation_predicted_2025.tif",
    meta_2025["profile"]
)


save_float_raster(
    confidence_2025,
    OUTPUT_DIR / "confidence_2025.tif",
    meta_2025["profile"]
)


print("2025 vegetation map saved.")

In [ ]:
# ============================================================
# 22. HIGH-CONFIDENCE CANDIDATE CHANGE
# ============================================================

valid_comparison = (
    (baseline_2011 > 0)
    &
    (prediction_2025 > 0)
)


different_class = (
    baseline_2011
    !=
    prediction_2025
)


# ------------------------------------------------------------
# Confidence requirement
# ------------------------------------------------------------

# If the 2011 value comes from the reference vegetation map,
# only the 2025 prediction confidence matters.

comparison_confidence = confidence_2025.copy()


# For reserves 8 and 9, BOTH 2011 and 2025 are modelled.
# Therefore use the lower confidence of the two dates.

modelled_2011 = (
    baseline_source == 2
)


comparison_confidence[
    modelled_2011
] = np.minimum(
    confidence_2011[modelled_2011],
    confidence_2025[modelled_2011]
)


# ------------------------------------------------------------
# Status codes
#
# 0 = NoData
# 1 = Same vegetation class
# 2 = Different class, but uncertain
# 3 = High-confidence candidate change
# ------------------------------------------------------------

change_status = np.zeros_like(
    baseline_2011
)


stable = (
    valid_comparison
    &
    (~different_class)
)


uncertain_change = (
    valid_comparison
    &
    different_class
    &
    (
        comparison_confidence
        <
        CONFIDENCE_THRESHOLD
    )
)


high_confidence_change = (
    valid_comparison
    &
    different_class
    &
    (
        comparison_confidence
        >=
        CONFIDENCE_THRESHOLD
    )
)


change_status[
    stable
] = 1


change_status[
    uncertain_change
] = 2


change_status[
    high_confidence_change
] = 3


save_class_raster(
    change_status,
    OUTPUT_DIR / "candidate_change_2011_2025.tif",
    meta_2011["profile"]
)


save_float_raster(
    comparison_confidence,
    OUTPUT_DIR / "change_confidence.tif",
    meta_2011["profile"]
)

In [ ]:
# ============================================================
# 23. VEGETATION AREA BY RESERVE
# ============================================================


def vegetation_area_table(
    vegetation_map,
    year
):

    rows = []


    for reserve_id in all_reserves:

        reserve_pixels = (
            reserve == reserve_id
        )


        for class_id, class_name in CLASS_NAMES.items():

            n_pixels = np.sum(
                reserve_pixels
                &
                (vegetation_map == class_id)
            )


            rows.append({
                "year": year,
                "reserve_id": reserve_id,
                "reserve": RESERVE_NAMES[reserve_id],
                "class_id": class_id,
                "vegetation_class": class_name,
                "pixels": n_pixels,
                "area_ha": n_pixels * PIXEL_AREA_HA
            })


    return pd.DataFrame(rows)



area_2011 = vegetation_area_table(
    baseline_2011,
    2011
)


area_2025 = vegetation_area_table(
    prediction_2025,
    2025
)


area_table = pd.concat(
    [
        area_2011,
        area_2025
    ],
    ignore_index=True
)


display(
    area_table
)


area_table.to_csv(
    OUTPUT_DIR / "vegetation_area_by_reserve.csv",
    index=False
)

In [ ]:
# ============================================================
# 24. 2011 → 2025 TRANSITION MATRIX
# ============================================================

from_class = baseline_2011[
    valid_comparison
]


to_class = prediction_2025[
    valid_comparison
]


transition = pd.crosstab(
    pd.Series(
        from_class,
        name="2011"
    ),
    pd.Series(
        to_class,
        name="2025"
    )
)


transition = transition.reindex(
    index=classes,
    columns=classes,
    fill_value=0
)


transition.index = [
    CLASS_NAMES[i]
    for i in transition.index
]


transition.columns = [
    CLASS_NAMES[i]
    for i in transition.columns
]


transition_ha = (
    transition
    *
    PIXEL_AREA_HA
)


print("Transition matrix — hectares")

display(
    transition_ha.round(2)
)


transition_ha.to_csv(
    OUTPUT_DIR / "vegetation_transition_2011_2025_ha.csv"
)

In [ ]:
# ============================================================
# 25. RESERVES 8 AND 9 CHANGE SUMMARY
# ============================================================

target_change = (
    target_mask
    &
    valid_comparison
)


n_target = np.sum(
    target_change
)


n_same = np.sum(
    target_change
    &
    stable
)


n_uncertain = np.sum(
    target_change
    &
    uncertain_change
)


n_high_confidence = np.sum(
    target_change
    &
    high_confidence_change
)


summary_targets = pd.DataFrame({
    "category": [
        "Comparable area",
        "Same predicted class",
        "Uncertain class difference",
        "High-confidence candidate change"
    ],

    "pixels": [
        n_target,
        n_same,
        n_uncertain,
        n_high_confidence
    ]
})


summary_targets[
    "area_ha"
] = (
    summary_targets["pixels"]
    *
    PIXEL_AREA_HA
)


display(
    summary_targets
)


summary_targets.to_csv(
    OUTPUT_DIR / "reserves_8_9_change_summary.csv",
    index=False
)